# Riemannian Median on SPD Matrices

This notebook runs the Riemannian median experiment on the manifold of Symmetric Positive Definite (SPD) matrices.
The objective is to compute the Fréchet median of a set of random SPD matrices:
$$f(P) = \frac{1}{N} \sum_{i=1}^{N} d(P, Q_i)$$
where $d$ is the geodesic distance on the SPD manifold and $Q_1, \dots, Q_N$ are the data points.

**Methods compared:** RPB, RPB-FO (first-order retraction), RCBM, RCBM-FO, PBA, PBA-FO, SGM, SGM-FO.

> **Note:** For the full multi-dimensional experiment (dims = [3, 5, 15, 30, 55]) that produces the paper figures, run `julia experiment_median_SPD.jl` from the repository root.

In [ ]:
using LinearAlgebra, Random, Statistics
using ManifoldDiff, Manifolds, Manopt, ManoptExamples
using LRUCache
using Plots

include("../src/RiemannianProximalBundle.jl")

# Inner product helper required by the RPB solver
inner_product(M, p, X, Y) = inner(M, p, X, Y)

## Parameters

Modify the values below to configure the experiment.

In [ ]:
# ── Experiment parameters ────────────────────────────────────────────────────
n             = 3        # SPD matrix dimension (e.g. 3, 5, 15, 30, 55)
N             = 20       # Number of data points
seed_argument = 57       # RNG seed for reproducibility
atol          = 1e-12    # Convergence tolerance
maxiter       = 1000     # Maximum iterations for each method

# ── RPB solver parameters ────────────────────────────────────────────────────
rpb_proximal_parameter = 1.0   # Initial proximal parameter ρ₀
rpb_trust_parameter    = 0.1   # Descent condition threshold β

# ── Curvature bounds for RCBM ────────────────────────────────────────────────
k_max = 0.0    # Upper sectional curvature bound
k_min = -0.5   # Lower sectional curvature bound

println("Configuration: n=$n, N=$N, maxiter=$maxiter, atol=$atol")

## Manifold, Objective, and Data Setup

In [ ]:
Random.seed!(seed_argument)

M    = SymmetricPositiveDefinite(n)
data = [rand(M) for _ in 1:N]

# Compute diameter and starting point
dists    = [distance(M, z, y) for z in data, y in data]
diameter = 2 * maximum(dists)
p0       = data[minimum(Tuple(findmax(dists)[2]))]

# Objective: mean geodesic distance to data points
f_spd(M, p)         = sum(1 / length(data) * distance.(Ref(M), Ref(p), data))
domf_spd(M, p)      = distance(M, p, p0) < diameter / 2
∂f_spd(M, p)        = sum(1/length(data) * ManifoldDiff.subgrad_distance.(Ref(M), data, Ref(p), 1; atol=atol))

# Retraction and transport maps
retraction_exp(x, v)          = exp(M, x, v)
transport_exp(x, y, v)        = vector_transport_to(M, x, v, y, ParallelTransport())

# First-order retraction: project p + v onto SPD cone
struct FirstOrderSPDRetraction <: AbstractRetractionMethod end
function Manifolds.retract!(::SymmetricPositiveDefinite, q, p, X, ::FirstOrderSPDRetraction)
    @. q = p + X
    q .= (q + q') / 2
    try
        cholesky(q)
    catch
        F = eigen(q)
        F.values .= max.(F.values, 1e-12)
        q .= F.vectors * Diagonal(F.values) * F.vectors'
    end
    return q
end
retraction_fo(x, v)           = retract(M, x, v, FirstOrderSPDRetraction())
transport_projection(x, y, v) = vector_transport_to(M, x, v, y, ProjectionTransport())

initial_obj    = f_spd(M, p0)
initial_subgrad = ∂f_spd(M, p0)
initial_entry  = (0, initial_obj, p0)

println("Initial objective: $initial_obj")

## Run Solvers

In [ ]:
# ── RPB (Exponential retraction + Parallel transport) ────────────────────────
println("Running RPB...")
rpb_solver = RProximalBundle(
    M, retraction_exp, transport_exp,
    (x) -> f_spd(M, x), (x) -> ∂f_spd(M, x),
    p0, initial_obj, initial_subgrad;
    max_iter=maxiter, tolerance=atol,
    proximal_parameter=rpb_proximal_parameter, trust_parameter=rpb_trust_parameter,
    know_minimizer=false, relative_error=false
)
run!(rpb_solver)
rpb_record = [(i, obj) for (i, obj) in enumerate(rpb_solver.raw_objective_history) .- 1]
println("  RPB: $(length(rpb_solver.indices_of_descent_steps)) descent steps, final obj = $(rpb_solver.raw_objective_history[end])")

# ── RPB-FO (First-order retraction + Projection transport) ───────────────────
println("Running RPB-FO...")
rpb_fo_solver = RProximalBundle(
    M, retraction_fo, transport_projection,
    (x) -> f_spd(M, x), (x) -> ∂f_spd(M, x),
    p0, initial_obj, initial_subgrad;
    max_iter=maxiter, tolerance=atol,
    retraction_error=1.0, transport_error=1.0,
    proximal_parameter=rpb_proximal_parameter, trust_parameter=rpb_trust_parameter,
    know_minimizer=false, relative_error=false
)
run!(rpb_fo_solver)
println("  RPB-FO: $(length(rpb_fo_solver.indices_of_descent_steps)) descent steps, final obj = $(rpb_fo_solver.raw_objective_history[end])")

# ── RCBM (Riemannian Convex Bundle Method from Manopt) ───────────────────────
println("Running RCBM...")
rcbm = convex_bundle_method(M, f_spd, ∂f_spd, p0;
    cache=(:LRU, [:Cost, :SubGradient], 50),
    domain=domf_spd, diameter=diameter, k_max=k_max, k_min=k_min,
    record=[:Iteration, :Cost, :Iterate], return_state=true,
    stopping_criterion=StopAfterIteration(maxiter)
)
rcbm_record = vcat([initial_entry], get_record(rcbm))
println("  RCBM: $(length(get_record(rcbm))) iterations, final obj = $(minimum([r[2] for r in get_record(rcbm)]))")

# ── SGM (Subgradient Method) ──────────────────────────────────────────────────
println("Running SGM...")
sgm = subgradient_method(M, f_spd, ∂f_spd, p0;
    cache=(:LRU, [:Cost, :SubGradient], 50),
    stepsize=DecreasingLength(; exponent=1, factor=1, subtrahend=0, length=1, shift=0, type=:absolute),
    record=[:Iteration, :Cost, :Iterate], return_state=true,
    stopping_criterion=StopWhenSubgradientNormLess(√atol) | StopAfterIteration(maxiter)
)
sgm_record = vcat([initial_entry], get_record(sgm))
println("  SGM: $(length(get_record(sgm))) iterations, final obj = $(minimum([r[2] for r in get_record(sgm)]))")

println("All solvers finished.")

## Results

In [ ]:
# Plot objective vs. iteration (oracle calls)
p_plot = plot(title="Riemannian Median on SPD($n×$n), N=$N",
              xlabel="Iteration", ylabel="Objective f(P)",
              yscale=:log10, legend=:topright, dpi=150)

rpb_iters  = 0:length(rpb_solver.raw_objective_history)-1
rpb_fo_iters = 0:length(rpb_fo_solver.raw_objective_history)-1
rcbm_iters = [r[1] for r in rcbm_record]
sgm_iters  = [r[1] for r in sgm_record]

plot!(p_plot, rpb_iters,    rpb_solver.raw_objective_history,    label="RPB",     color=:blue,   linewidth=2)
plot!(p_plot, rpb_fo_iters, rpb_fo_solver.raw_objective_history, label="RPB-FO",  color=:cyan,   linewidth=2, linestyle=:dash)
plot!(p_plot, rcbm_iters,   [r[2] for r in rcbm_record],         label="RCBM",    color=:red,    linewidth=2)
plot!(p_plot, sgm_iters,    [r[2] for r in sgm_record],          label="SGM",     color=:orange, linewidth=1)

display(p_plot)